<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 09. Attention Is All You Need — Attention Is All You Need

- **원 논문:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- **저자 / 발표:** Ashish Vaswani et al. · NeurIPS 2017 (2017)
- **난이도 / 예상 시간:** 중상급 · 약 80분
- **선수 지식:** 선형대수, softmax, broadcasting, attention mask

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

식 (1)의 scaled dot-product attention, 식 (2)--(3)의 multi-head projection, §3.3의 position-wise FFN, §3.1의 residual connection과 post-LayerNorm, §3.5의 sinusoidal positional encoding을 직접 구현한다. 같은 block을 encoder처럼 mask 없이, decoder self-attention처럼 causal mask와 함께 실행해 미래 token 불변성, 확률 정규화, scale이 entropy에 미치는 효과를 검증한다.

**원 논문과 다른 것**

원 논문의 base Transformer는 6층 encoder/decoder, d_model=512, 8 heads로 WMT'14 En-De 약 450만 문장 쌍을 학습했다. 이 노트북은 d_model=16, 4 heads인 단일 self-attention+FFN block을 사용한다. encoder-decoder cross-attention, 6층 stack, label smoothing, 번역 학습과 beam search는 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V | `scaled_dot_product_attention`의 score→mask→softmax→weighted sum | attention 행 합, shape, 유한값을 assert |
| §3.2.2, 식 (2)--(3): head별 projection 후 concat과 W^O | `MultiHeadSelfAttention`의 projection/split/transpose/merge | 출력과 head별 weight shape 검사 |
| §3.1: decoder self-attention에서 subsequent position 차단 | `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)` | 미래 attention mass=0 및 prefix 출력 불변성 assert |
| §3.5: sinusoidal positional encoding 수식 | `sinusoidal_positions`의 짝수 sin/홀수 cos 구현 | position 0 패턴과 shape 검사 |
| §3.2.1: 1/√d_k scaling이 큰 내적으로 인한 작은 gradient를 완화 | `attention_entropy`의 scaled/unscaled score 비교 | scaled entropy가 더 큰지 정량 비교하고 막대그래프로 표시 |
| §3.1: Sublayer 출력은 LayerNorm(x + Sublayer(x)) | `TransformerBlock`의 attention/FFN residual 뒤 각각 post-LayerNorm | residual shape와 token별 평균·분산을 검사 |
| §3.3, 식 FFN(x)=max(0,xW1+b1)W2+b2 | `PositionWiseFeedForward`의 두 Linear와 ReLU | 동일 module이 모든 token 위치에 독립 적용되는 shape를 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\operatorname{Attention}(Q,K,V)=
\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V
$$

- $Q,K,V$는 query·key·value, $d_k$는 head dimension, $M$은 causal/padding mask입니다.
- $1/\sqrt{d_k}$ scaling은 dot-product variance를 줄여 softmax saturation을 방지합니다.
- projection, head split/merge, mask, attention weight를 순서대로 직접 구현합니다.
- $[B,T,D]\rightarrow[B,H,T,d_k]\rightarrow[B,T,D]$의 shape 변화를 assertion으로 확인합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V
- **노트북 구현:** `scaled_dot_product_attention`의 score→mask→softmax→weighted sum
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** attention 행 합, shape, 유한값을 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문의 base Transformer는 6층 encoder/decoder, d_model=512, 8 heads로 WMT'14 En-De 약 450만 문장 쌍을 학습했다. 이 노트북은 d_model=16, 4 heads인 단일 self-attention+FFN block을 사용한다. encoder-decoder cross-attention, 6층 stack, label smoothing, 번역 학습과 beam search는 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

Transformer는 recurrence 없이 attention만으로 위치 간 정보를 섞는다.
dot product의 분산은 차원 $d_k$와 함께 커지므로 식 (1)은 점수를
$\sqrt{d_k}$로 나눈다. 여러 projection 공간에서 병렬로 attention한 결과를
이어 붙이는 것이 multi-head attention이다. 순서는 고정 sin/cos 위치 벡터를
token 표현에 더해 제공하며, decoder에는 미래를 볼 수 없는 causal mask가 필요하다.

§3.1의 원 논문 block은 각 sublayer 뒤에
$\mathrm{LayerNorm}(x+\mathrm{Sublayer}(x))$를 적용하는 **post-LN** 구조다.
여기서도 이 순서를 그대로 드러내며, §3.3의 position-wise FFN도 고수준
Transformer module 안에 숨기지 않는다. `allowed=None`은 encoder self-attention,
lower-triangular mask는 decoder의 첫 masked self-attention sublayer에 대응한다.
단, decoder의 encoder-decoder attention은 이 축소 실험 범위 밖이다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
from llm_engineering_lab.acceleration import get_accelerator

SEED = 909
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.5: sinusoidal positional encoding 수식
- **이 셀의 책임:** `sinusoidal_positions`의 짝수 sin/홀수 cos 구현
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** position 0 패턴과 shape 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.arange,torch.float32,torch.exp,torch.log,torch.zeros,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>torch.exp(input)</code></summary>

#### `torch.exp(input)`

- **역할:** 각 원소에 자연지수함수 $e^x$를 적용합니다.
- **입력·반환:** 실수 텐서를 받고 같은 shape의 양수 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit, 확률비, diffusion coefficient를 양수 scale로 바꿉니다.
- **주의점:** 큰 양수는 overflow, 큰 음수는 0에 가까운 underflow를 만들 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=e^{x_i}
$$

- **기호:** $e$는 자연로그의 밑입니다.
- **수식 → 코드:** API가 각 원소를 양수 값으로 지수 변환합니다.
- **직관:** 더하기 차이를 곱셈 비율로 바꾸며 log-domain 값을 원래 scale로 복원합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.exp](https://docs.pytorch.org/docs/stable/generated/torch.exp.html)

</details>

<details>
<summary><code>torch.log(input)</code></summary>

#### `torch.log(input)`

- **역할:** 각 원소에 자연로그를 적용합니다.
- **입력·반환:** 양수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 곱을 합으로 바꾸고 log-likelihood를 계산합니다.
- **주의점:** 0은 음의 무한대, 음수는 NaN이므로 작은 epsilon이나 clamp가 필요합니다.

##### 관련 수식과 코드 연결

$$
y_i=\log x_i,\qquad x_i>0
$$

- **기호:** $\log$는 자연로그입니다.
- **수식 → 코드:** API가 곱셈 관계를 덧셈 관계로 바꾸고 작은 확률의 크기를 안정적으로 표현합니다.
- **직관:** 확률 곱을 log-probability 합으로 계산할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같으며 입력은 양수여야 합니다.
- **공식 문서:** [torch.log](https://docs.pytorch.org/docs/stable/generated/torch.log.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 09-1: 논문 §3.5의 짝수 sin/홀수 cos positional encoding을 구현하세요.
def sinusoidal_positions(length, d_model):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


positions = sinusoidal_positions(length=6, d_model=16)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V
- **이 셀의 책임:** `scaled_dot_product_attention`의 score→mask→softmax→weighted sum
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** attention 행 합, shape, 유한값을 assert. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.sqrt,Tensor.transpose,Tensor.masked_fill,torch.softmax,torch.randn,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 6개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>torch.softmax(input, dim, dtype=None)</code></summary>

#### `torch.softmax(input, dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** attention score나 class logit을 정규화합니다.
- **주의점:** `dim`을 잘못 고르면 엉뚱한 축이 정규화되고 큰 logit은 포화될 수 있습니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 09-2: 식 (1)의 scale, mask, softmax, value 가중합을 구현하세요.
def scaled_dot_product_attention(query, key, value, allowed=None):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2.1, 식 (1): Attention(Q,K,V)=softmax(QKᵀ/√d_k)V / §3.2.2, 식 (2)--(3): head별 projection 후 concat과 W^O / §3.1: decoder self-attention에서 subsequent position 차단
- **이 셀의 책임:** `scaled_dot_product_attention`의 score→mask→softmax→weighted sum / `MultiHeadSelfAttention`의 projection/split/transpose/merge / `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)`
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** attention 행 합, shape, 유한값을 assert / 출력과 head별 weight shape 검사 / 미래 attention mass=0 및 prefix 출력 불변성 assert. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,Tensor.view,torch.ones,torch.bool -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>tensor.view(*shape)</code></summary>

#### `tensor.view(*shape)`

- **역할:** 같은 저장공간을 보는 새로운 shape의 텐서를 만듭니다.
- **입력·반환:** 새 shape을 받고 view 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 연속 메모리의 축을 복사 없이 재해석합니다.
- **주의점:** stride가 호환되지 않으면 실패하므로 `reshape`나 `contiguous`가 필요합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.view](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 09-3: 식 (2)--(3)의 head split, attention, concat, W^O를 구현하세요.
def causal_attention_mask(length, device):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


class MultiHeadSelfAttention(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, num_heads):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def _split(self, tensor):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def _merge(self, tensor):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, hidden, allowed=None):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.2.2, 식 (2)--(3): head별 projection 후 concat과 W^O / §3.1: decoder self-attention에서 subsequent position 차단 / §3.1: Sublayer 출력은 LayerNorm(x + Sublayer(x)) / §3.3, 식 FFN(x)=max(0,xW1+b1)W2+b2
- **이 셀의 책임:** `MultiHeadSelfAttention`의 projection/split/transpose/merge / `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)` / `TransformerBlock`의 attention/FFN residual 뒤 각각 post-LayerNorm / `PositionWiseFeedForward`의 두 Linear와 ReLU
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 출력과 head별 weight shape 검사 / 미래 attention mass=0 및 prefix 출력 불변성 assert / residual shape와 token별 평균·분산을 검사 / 동일 module이 모든 token 위치에 독립 적용되는 shape를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=F.relu,nn.LayerNorm,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>F.relu(input, inplace=False)</code></summary>

#### `F.relu(input, inplace=False)`

- **역할:** 음수 activation을 0으로 만드는 ReLU를 적용합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 선형 층 사이에 비선형성을 추가합니다.
- **주의점:** 음수 구간 gradient는 0이며 in-place 연산은 autograd 값을 덮어쓸 수 있습니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [F.relu](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.relu.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 09-4: §3.3의 FFN과 §3.1의 post-LN residual block을 구현하세요.
class PositionWiseFeedForward(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, d_ff):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TransformerBlock(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, d_model, num_heads, d_ff):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden, allowed=None):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


block = TransformerBlock(d_model=16, num_heads=4, d_ff=32).to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1: decoder self-attention에서 subsequent position 차단 / §3.2.1: 1/√d_k scaling이 큰 내적으로 인한 작은 gradient를 완화
- **이 셀의 책임:** `causal_attention_mask`와 `TransformerBlock(..., allowed=causal)` / `attention_entropy`의 scaled/unscaled score 비교
- **Tensor shape 계약:** hidden [B,T,D] ↔ heads [B,H,T,dₖ] → block output [B,T,D]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 미래 attention mass=0 및 prefix 출력 불변성 assert / scaled entropy가 더 큰지 정량 비교하고 막대그래프로 표시. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode,Tensor.softmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>tensor.softmax(dim, dtype=None)</code></summary>

#### `tensor.softmax(dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** method 문법으로 attention score나 class logit을 정규화합니다.
- **주의점:** `dim`이 확률 합을 낼 축이며 입력은 일반적으로 raw logit이어야 합니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [Tensor.softmax](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.softmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO 09-5: full block에서도 미래 token 불변성을 검증하세요.
# TODO 09-6: scaled/unscaled score entropy를 비교하고 attention을 그리세요.
def attention_entropy(weights):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


changed_hidden = hidden.clone()
changed_hidden[:, -1] += 100.0
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 해석 체크

1. `sqrt(d_k)`로 나누지 않으면 softmax와 gradient에 어떤 일이 생기나요?
2. 원 논문의 post-LN 수식과 최신 pre-LN 구현의 연산 순서는 어떻게 다른가요?
3. causal mask 검증에서 마지막 token을 크게 바꾼 이유는 무엇인가요?
4. 서로 다른 head가 유용할 수 있는 이유를 projection 관점에서 설명하세요.

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.